# Exemplo: Backend Ray
-----------------------

Este exemplo mostra como usar o backend Ray para treinar modelos em um contexto paralelo.

Os dados utilizados são um conjunto sintético criado com a função [make_classification](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html) do sklearn.

## Carregar os dados

In [1]:
# Import packages
import ray
from experionml import ExperionMLClassifier
from sklearn.datasets import make_classification

ModuleNotFoundError: No module named 'ray'

In [ ]:
# Use um conjunto de dados pequeno para fins ilustrativos
X, y = make_classification(n_samples=10000, n_features=10, random_state=1)

## Executar o pipeline

In [ ]:
# Observe que já especificamos aqui o número de núcleos para execução paralela
experionml = ExperionMLClassifier(X, y, n_jobs=2, backend="ray", verbose=2, random_state=1)

In [ ]:
# Com `parallel=True`, treinamos um modelo em cada nó
# Observe que, ao treinar em paralelo, a verbosidade dos modelos é zero
experionml.run(models=["PA", "SGD"], est_params={"max_iter": 150}, parallel=True, errors="raise")

## Analisar os resultados

In [ ]:
# Observe como a soma do tempo para treinar os modelos é menor que o tempo total
experionml.plot_results(metric="time_fit")

In [ ]:
# Crie um endpoint de API REST e faça inferência no conjunto holdout
experionml.pa.serve(port=8001)

In [ ]:
import requests

X_predict = experionml.X_test.iloc[:10, :]
response = requests.get("http://127.0.0.1:8001/", json=X_predict.to_json())

In [ ]:
response.json()

In [ ]:
# Não se esqueça de desligar o servidor Ray
ray.shutdown()